# 7. N-Step Bootstrapping

Bu notebook, Sutton & Barto kitabının 7. bölümünü kapsar.

## İçindekiler
1. N-Step TD Prediction
2. N-Step SARSA
3. N-Step Off-policy Learning
4. N-Step Tree Backup

## 7.1 TD ve MC Arasındaki Spektrum

TD(0) ve Monte Carlo, bir spektrumun iki ucudur:

| Yöntem | Kaç adım bekler? | Bootstrap? |
|--------|-----------------|------------|
| TD(0) | 1 adım | Evet |
| 2-step TD | 2 adım | Evet |
| n-step TD | n adım | Evet |
| MC | Episode sonu | Hayır |

### N-Step Return

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

- $n=1$: $G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$ → TD(0)
- $n=\infty$: $G_t^{(\infty)} = R_{t+1} + \gamma R_{t+2} + ...$ → MC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

class RandomWalkEnv:
    """
    19-state Random Walk (Example 7.1).
    States: 0 (left terminal), 1-19, 20 (right terminal)
    Start: state 10 (middle)
    Left terminal: reward 0
    Right terminal: reward 1
    """
    
    def __init__(self, n_states=19):
        self.n_states = n_states
        self.start_state = n_states // 2 + 1  # Middle state
        self.left_terminal = 0
        self.right_terminal = n_states + 1
        self.reset()
    
    def reset(self):
        self.state = self.start_state
        return self.state
    
    def step(self, action=None):
        """Random walk: 50% left, 50% right."""
        if np.random.random() < 0.5:
            self.state -= 1  # Left
        else:
            self.state += 1  # Right
        
        # Check terminals
        if self.state == self.left_terminal:
            return self.state, -1, True
        elif self.state == self.right_terminal:
            return self.state, 1, True
        else:
            return self.state, 0, False

# True values for random walk
def compute_true_values(n_states=19):
    """True state values for random walk."""
    # V(s) = (s - 0) / (n_states + 1) normalized to [-1, 1]
    return np.array([(2*s - n_states - 1) / (n_states + 1) for s in range(1, n_states + 1)])

env = RandomWalkEnv()
true_values = compute_true_values()

print(f"States: 1-{env.n_states}")
print(f"Start state: {env.start_state}")
print(f"True values: {true_values.round(2)}")

In [ ]:
# Visualize random walk
plt.figure(figsize=(12, 3))
states = range(1, env.n_states + 1)

plt.bar(states, true_values, color='steelblue', alpha=0.7)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('State')
plt.ylabel('True Value V(s)')
plt.title('19-State Random Walk: True Values')
plt.xticks(states)
plt.grid(True, alpha=0.3)
plt.show()

## 7.2 N-Step TD Prediction

### Update Rule

$$V(S_t) \leftarrow V(S_t) + \alpha [G_t^{(n)} - V(S_t)]$$

where:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

### Algoritma

N-step TD, güncellemeleri **n adım gecikmeyle** yapar.

In [ ]:
def n_step_td_prediction(env, n, n_episodes=10, alpha=0.1, gamma=1.0):
    """
    N-step TD Prediction.
    
    Args:
        env: Environment
        n: Number of steps
        n_episodes: Number of episodes
        alpha: Learning rate
        gamma: Discount factor
    
    Returns:
        V: State value estimates
    """
    V = np.zeros(env.n_states + 2)  # Include terminals
    
    for episode in range(n_episodes):
        # Store states and rewards
        states = [env.reset()]
        rewards = [0]  # R_0 is not used
        
        T = float('inf')  # Terminal time
        t = 0
        
        while True:
            if t < T:
                next_state, reward, done = env.step()
                states.append(next_state)
                rewards.append(reward)
                
                if done:
                    T = t + 1
            
            # Update time
            tau = t - n + 1  # Time of state being updated
            
            if tau >= 0:
                # Calculate n-step return
                G = 0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                
                if tau + n < T:
                    G += (gamma ** n) * V[states[tau + n]]
                
                # Update
                s = states[tau]
                if 1 <= s <= env.n_states:  # Non-terminal
                    V[s] += alpha * (G - V[s])
            
            if tau == T - 1:
                break
            
            t += 1
    
    return V[1:env.n_states + 1]  # Return non-terminal values

# Test different n values
n_values = [1, 2, 4, 8, 16]
n_episodes = 10

results = {}
for n in n_values:
    V = n_step_td_prediction(env, n, n_episodes=n_episodes, alpha=0.4)
    rmse = np.sqrt(np.mean((V - true_values) ** 2))
    results[n] = {'V': V, 'rmse': rmse}
    print(f"n={n}: RMSE = {rmse:.4f}")

In [ ]:
# Compare different n values
def run_n_step_experiment(env, n_values, alphas, n_episodes=10, n_runs=100):
    """Run experiments for different n and alpha values."""
    
    true_v = compute_true_values()
    results = np.zeros((len(n_values), len(alphas)))
    
    for i, n in enumerate(n_values):
        for j, alpha in enumerate(alphas):
            errors = []
            for _ in range(n_runs):
                V = n_step_td_prediction(env, n, n_episodes=n_episodes, alpha=alpha)
                rmse = np.sqrt(np.mean((V - true_v) ** 2))
                errors.append(rmse)
            results[i, j] = np.mean(errors)
    
    return results

n_values = [1, 2, 4, 8, 16, 32]
alphas = np.linspace(0.1, 1.0, 10)

# This may take a moment
print("Running experiments...")
rmse_results = run_n_step_experiment(env, n_values, alphas, n_episodes=10, n_runs=50)

# Plot
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(n_values)))

for i, n in enumerate(n_values):
    plt.plot(alphas, rmse_results[i], label=f'n={n}', color=colors[i], linewidth=2)

plt.xlabel('α (step size)')
plt.ylabel('Average RMSE')
plt.title('N-Step TD: Effect of n and α')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7.3 N-Step SARSA

N-step fikrini control'e uygula:

### N-Step Return (Q için)

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n Q(S_{t+n}, A_{t+n})$$

### Update

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [G_t^{(n)} - Q(S_t, A_t)]$$

In [ ]:
class GridWorldEnv:
    """Simple grid world for n-step SARSA demo."""
    
    def __init__(self, rows=4, cols=4):
        self.rows = rows
        self.cols = cols
        self.start = (3, 0)
        self.goal = (0, 3)
        self.n_actions = 4
        
        self.actions = {
            0: (-1, 0),  # up
            1: (0, 1),   # right
            2: (1, 0),   # down
            3: (0, -1)   # left
        }
        self.reset()
    
    def reset(self):
        self.position = self.start
        return self.position
    
    def step(self, action):
        row, col = self.position
        drow, dcol = self.actions[action]
        
        new_row = max(0, min(self.rows - 1, row + drow))
        new_col = max(0, min(self.cols - 1, col + dcol))
        self.position = (new_row, new_col)
        
        if self.position == self.goal:
            return self.position, 0, True
        return self.position, -1, False

env_grid = GridWorldEnv()

In [ ]:
def n_step_sarsa(env, n, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    N-step SARSA.
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    episode_lengths = []
    
    for episode in range(n_episodes):
        states = [env.reset()]
        actions = [epsilon_greedy(states[0])]
        rewards = [0]
        
        T = float('inf')
        t = 0
        
        while True:
            if t < T:
                next_state, reward, done = env.step(actions[t])
                states.append(next_state)
                rewards.append(reward)
                
                if done:
                    T = t + 1
                else:
                    actions.append(epsilon_greedy(next_state))
            
            tau = t - n + 1
            
            if tau >= 0:
                G = 0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                
                if tau + n < T:
                    G += (gamma ** n) * Q[states[tau + n]][actions[tau + n]]
                
                s, a = states[tau], actions[tau]
                Q[s][a] += alpha * (G - Q[s][a])
            
            if tau == T - 1:
                break
            
            t += 1
        
        episode_lengths.append(T)
    
    return Q, episode_lengths

# Compare different n values
n_values = [1, 2, 4, 8]
results_sarsa = {}

for n in n_values:
    Q, lengths = n_step_sarsa(env_grid, n, n_episodes=200)
    results_sarsa[n] = lengths
    print(f"n={n}: Average episode length (last 50) = {np.mean(lengths[-50:]):.1f}")

In [ ]:
# Plot learning curves
plt.figure(figsize=(10, 5))

window = 10
for n in n_values:
    smoothed = np.convolve(results_sarsa[n], np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=f'n={n}', linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Steps per Episode')
plt.title('N-Step SARSA: Learning Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7.4 N-Step Off-policy Learning

Off-policy n-step learning, importance sampling gerektirir:

### Importance Sampling Ratio

$$\rho_{t:h} = \prod_{k=t}^{\min(h, T-1)} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

### N-Step Off-policy Update

$$V(S_t) \leftarrow V(S_t) + \alpha \rho_{t:t+n-1} [G_t^{(n)} - V(S_t)]$$

In [ ]:
def n_step_off_policy_sarsa(env, n, n_episodes=500, alpha=0.5, gamma=1.0, 
                            epsilon_behavior=0.1, epsilon_target=0.0):
    """
    N-step Off-policy SARSA with Importance Sampling.
    
    behavior policy: ε-greedy with epsilon_behavior
    target policy: ε-greedy with epsilon_target (0 = greedy)
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def behavior_policy(state):
        if np.random.random() < epsilon_behavior:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    def target_policy_prob(state, action):
        """π(a|s) for target policy."""
        best_action = np.argmax(Q[state])
        if action == best_action:
            return 1 - epsilon_target + epsilon_target / env.n_actions
        return epsilon_target / env.n_actions
    
    def behavior_policy_prob(state, action):
        """b(a|s) for behavior policy."""
        best_action = np.argmax(Q[state])
        if action == best_action:
            return 1 - epsilon_behavior + epsilon_behavior / env.n_actions
        return epsilon_behavior / env.n_actions
    
    episode_lengths = []
    
    for episode in range(n_episodes):
        states = [env.reset()]
        actions = [behavior_policy(states[0])]
        rewards = [0]
        
        T = float('inf')
        t = 0
        
        while True:
            if t < T:
                next_state, reward, done = env.step(actions[t])
                states.append(next_state)
                rewards.append(reward)
                
                if done:
                    T = t + 1
                else:
                    actions.append(behavior_policy(next_state))
            
            tau = t - n + 1
            
            if tau >= 0:
                # Calculate importance sampling ratio
                rho = 1.0
                for k in range(tau + 1, min(tau + n, T)):
                    pi_prob = target_policy_prob(states[k], actions[k])
                    b_prob = behavior_policy_prob(states[k], actions[k])
                    rho *= pi_prob / b_prob
                
                # Calculate n-step return
                G = 0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                
                if tau + n < T:
                    G += (gamma ** n) * Q[states[tau + n]][actions[tau + n]]
                
                # Update with importance sampling
                s, a = states[tau], actions[tau]
                Q[s][a] += alpha * rho * (G - Q[s][a])
            
            if tau == T - 1:
                break
            
            t += 1
        
        episode_lengths.append(T)
    
    return Q, episode_lengths

# Test
Q_off, lengths_off = n_step_off_policy_sarsa(env_grid, n=4, n_episodes=200)
print(f"Off-policy n-step SARSA: Avg length = {np.mean(lengths_off[-50:]):.1f}")

## 7.5 N-Step Tree Backup

Importance sampling olmadan off-policy learning:

**Fikir**: Sadece alınan aksiyonları değil, **tüm aksiyonların değerlerini** kullan.

### Tree Backup Return

$$G_t^{(n)} = R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1}) Q(S_{t+1}, a) + \gamma \pi(A_{t+1}|S_{t+1}) G_{t+1}^{(n-1)}$$

Bu, bir **ağaç yapısı** oluşturur: alınan aksiyonun branch'ı devam eder, diğerleri "backup" edilir.

In [ ]:
def n_step_tree_backup(env, n, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    N-step Tree Backup Algorithm.
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def policy_probs(state):
        """ε-greedy action probabilities."""
        probs = np.ones(env.n_actions) * epsilon / env.n_actions
        best_action = np.argmax(Q[state])
        probs[best_action] += 1 - epsilon
        return probs
    
    def select_action(state):
        probs = policy_probs(state)
        return np.random.choice(env.n_actions, p=probs)
    
    episode_lengths = []
    
    for episode in range(n_episodes):
        states = [env.reset()]
        actions = [select_action(states[0])]
        rewards = [0]
        
        T = float('inf')
        t = 0
        
        while True:
            if t < T:
                next_state, reward, done = env.step(actions[t])
                states.append(next_state)
                rewards.append(reward)
                
                if done:
                    T = t + 1
                else:
                    actions.append(select_action(next_state))
            
            tau = t - n + 1
            
            if tau >= 0:
                # Tree backup calculation
                if t + 1 >= T:
                    G = rewards[T]
                else:
                    probs = policy_probs(states[t + 1])
                    G = rewards[t + 1] + gamma * np.dot(probs, Q[states[t + 1]])
                
                for k in range(min(t, T - 1), tau, -1):
                    probs = policy_probs(states[k])
                    G = rewards[k] + gamma * np.sum(
                        probs * Q[states[k]] * (np.arange(env.n_actions) != actions[k])
                    ) + gamma * probs[actions[k]] * G
                
                s, a = states[tau], actions[tau]
                Q[s][a] += alpha * (G - Q[s][a])
            
            if tau == T - 1:
                break
            
            t += 1
        
        episode_lengths.append(T)
    
    return Q, episode_lengths

# Test
Q_tree, lengths_tree = n_step_tree_backup(env_grid, n=4, n_episodes=200)
print(f"Tree Backup: Avg length = {np.mean(lengths_tree[-50:]):.1f}")

In [ ]:
# Compare all n-step methods
methods = {
    'n-step SARSA': lambda: n_step_sarsa(env_grid, n=4, n_episodes=200)[1],
    'Off-policy': lambda: n_step_off_policy_sarsa(env_grid, n=4, n_episodes=200)[1],
    'Tree Backup': lambda: n_step_tree_backup(env_grid, n=4, n_episodes=200)[1]
}

plt.figure(figsize=(10, 5))
window = 10

for name, method in methods.items():
    lengths = method()
    smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=name, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Steps per Episode')
plt.title('N-Step Methods Comparison (n=4)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Özet

| Yöntem | n | Bootstrap | Variance | Bias |
|--------|---|-----------|----------|------|
| TD(0) | 1 | Yüksek | Düşük | Yüksek |
| n-step TD | n | Orta | Orta | Orta |
| MC | ∞ | Yok | Yüksek | Düşük |

### N-Step Yöntemleri

| Yöntem | Tip | Özellik |
|--------|-----|--------|
| **n-step TD** | Prediction | Basit n-step return |
| **n-step SARSA** | On-policy Control | Q için n-step |
| **n-step Off-policy** | Off-policy | Importance sampling gerekli |
| **Tree Backup** | Off-policy | IS gereksiz, daha kararlı |

### Optimal n Seçimi
- Ortam ve görev bağımlı
- Genelde n=4 veya n=8 iyi çalışır
- Çok büyük n → MC'ye yaklaşır (yüksek variance)
- Çok küçük n → TD'ye yaklaşır (yüksek bias)

### Sonraki Notebook
**08 - Planning and Learning**: Model-based methods, Dyna-Q